In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import scipy.io
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import torch

DATA_PATH = "/content/drive/MyDrive/cylinder_nektar_wake.mat"
SEED = 42
TRAIN_T_MAX = 18.0
RE = 100.0

N_SENSORS = 200
N_PAIRS = 3000
POINTS_PER_PAIR = 300
N_PHYS_PAIRS = 3000
PHYS_PTS_PER_PAIR = 100

PINN_ITERS = 12000
PINN_LR = 1e-3
LAMBDA_PHYS_PINN = 1.0

DON_ITERS = 10000
DON_LR = 1e-4

PI_ITERS = 12000
PI_LR = 1e-4
SA_LR = 5e-3

BATCH = 1000
RUN_TAG = "v23_train_to_18"

np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("switch colab runtime to GPU first")
device = "cuda"

raw = scipy.io.loadmat(DATA_PATH)
X_star = raw["X_star"]
U_star = raw["U_star"]
p_star = raw["p_star"]
t_key = next(k for k in ("t_star", "t", "T_star", "tt") if k in raw)
t_star = raw[t_key].flatten()

u_all, v_all = U_star[:, 0, :], U_star[:, 1, :]
p_all = p_star
N, T = u_all.shape

train_mask = t_star <= TRAIN_T_MAX
train_idx = np.where(train_mask)[0]
test_idx = np.where(~train_mask)[0]
split_i = train_idx[-1]

x_min, x_max = X_star[:, 0].min(), X_star[:, 0].max()
y_min, y_max = X_star[:, 1].min(), X_star[:, 1].max()
t_min, t_max = t_star.min(), t_star.max()
dt_max = t_max - t_min

u_min, u_max = u_all[:, train_idx].min(), u_all[:, train_idx].max()
v_min, v_max = v_all[:, train_idx].min(), v_all[:, train_idx].max()
p_min, p_max = p_all[:, train_idx].min(), p_all[:, train_idx].max()

X_SPAN, Y_SPAN, T_SPAN = x_max - x_min, y_max - y_min, t_max - t_min
U_SPAN, V_SPAN, P_SPAN = u_max - u_min, v_max - v_min, p_max - p_min

def norm(a, lo, hi): return (a - lo) / (hi - lo)
def denorm(a, lo, hi): return a * (hi - lo) + lo

u_n = norm(u_all, u_min, u_max)
v_n = norm(v_all, v_min, v_max)
p_n = norm(p_all, p_min, p_max)
x_n = norm(X_star[:, 0], x_min, x_max)
y_n = norm(X_star[:, 1], y_min, y_max)
t_n_all = norm(t_star, t_min, t_max)

def rel_l2(pred, true):
    return np.linalg.norm(pred - true) / np.linalg.norm(true)

def mlp(sizes, act):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(torch.nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return torch.nn.Sequential(*layers)

def pinn_residual(net_fn, coords):
    coords = coords.detach().clone().requires_grad_(True)
    out = net_fn(coords)
    u = out[:, 0:1] * U_SPAN + u_min
    v = out[:, 1:2] * V_SPAN + v_min
    p = out[:, 2:3] * P_SPAN + p_min

    def grad(y, x):
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y),
                                   create_graph=True, retain_graph=True)[0]

    du = grad(u, coords); dv = grad(v, coords); dp = grad(p, coords)
    u_x = du[:, 0:1] / X_SPAN; u_y = du[:, 1:2] / Y_SPAN; u_t = du[:, 2:3] / T_SPAN
    v_x = dv[:, 0:1] / X_SPAN; v_y = dv[:, 1:2] / Y_SPAN; v_t = dv[:, 2:3] / T_SPAN
    p_x = dp[:, 0:1] / X_SPAN; p_y = dp[:, 1:2] / Y_SPAN

    u_xx = grad(u_x, coords)[:, 0:1] / X_SPAN
    u_yy = grad(u_y, coords)[:, 1:2] / Y_SPAN
    v_xx = grad(v_x, coords)[:, 0:1] / X_SPAN
    v_yy = grad(v_y, coords)[:, 1:2] / Y_SPAN

    r_cont = u_x + v_y
    r_xmom = u_t + u * u_x + v * u_y + p_x - (1.0 / RE) * (u_xx + u_yy)
    r_ymom = v_t + u * v_x + v * v_y + p_y - (1.0 / RE) * (v_xx + v_yy)
    return r_cont, r_xmom, r_ymom

n_tr = len(train_idx)
pinn_X = np.empty((N * n_tr, 3), dtype=np.float32)
pinn_Y = np.empty((N * n_tr, 3), dtype=np.float32)
for k, it in enumerate(train_idx):
    s, e = k * N, (k + 1) * N
    pinn_X[s:e, 0] = x_n
    pinn_X[s:e, 1] = y_n
    pinn_X[s:e, 2] = t_n_all[it]
    pinn_Y[s:e, 0] = u_n[:, it]
    pinn_Y[s:e, 1] = v_n[:, it]
    pinn_Y[s:e, 2] = p_n[:, it]

pinn_net = mlp([3, 128, 128, 128, 128, 128, 128, 3], torch.nn.Tanh).to(device)
pinn_opt = torch.optim.Adam(pinn_net.parameters(), lr=PINN_LR)

pXt = torch.tensor(pinn_X, device=device)
pYt = torch.tensor(pinn_Y, device=device)

coll = np.random.rand(100000, 3).astype(np.float32)
coll[:, 2] *= norm(TRAIN_T_MAX, t_min, t_max)
pColl = torch.tensor(coll, device=device)

print("model 1/3, training pinn on t 0 to 18")
for step in range(PINN_ITERS + 1):
    idx_d = torch.randint(0, pXt.shape[0], (BATCH,))
    idx_c = torch.randint(0, pColl.shape[0], (BATCH,))

    pred = pinn_net(pXt[idx_d])
    loss_data = ((pred - pYt[idx_d]) ** 2).mean()

    rc, rx, ry = pinn_residual(pinn_net, pColl[idx_c])
    loss_phys = (rc ** 2).mean() + (rx ** 2).mean() + (ry ** 2).mean()

    loss = loss_data + LAMBDA_PHYS_PINN * loss_phys
    pinn_opt.zero_grad()
    loss.backward()
    pinn_opt.step()

    if step % 2000 == 0:
        print(f"pinn step {step:5d} | data {loss_data.item():.3e} | phys {loss_phys.item():.3e}")

sensor_idx = np.random.choice(N, N_SENSORS, replace=False)

def make_branch(i_in):
    return np.concatenate([u_n[sensor_idx, i_in], v_n[sensor_idx, i_in]])

def build_dataset(n_pairs, pts_per_pair, time_pool, rng):
    B = np.empty((n_pairs * pts_per_pair, 2 * N_SENSORS), dtype=np.float32)
    Tr = np.empty((n_pairs * pts_per_pair, 4), dtype=np.float32)
    Y = np.empty((n_pairs * pts_per_pair, 3), dtype=np.float32)
    for k in range(n_pairs):
        i_in, i_out = np.sort(rng.choice(time_pool, 2, replace=False))
        pts = rng.choice(N, pts_per_pair, replace=False)
        s, e = k * pts_per_pair, (k + 1) * pts_per_pair
        B[s:e] = make_branch(i_in).astype(np.float32)
        Tr[s:e, 0] = x_n[pts]
        Tr[s:e, 1] = y_n[pts]
        Tr[s:e, 2] = t_n_all[i_out]
        Tr[s:e, 3] = (t_star[i_out] - t_star[i_in]) / dt_max
        Y[s:e, 0] = u_n[pts, i_out]
        Y[s:e, 1] = v_n[pts, i_out]
        Y[s:e, 2] = p_n[pts, i_out]
    return B, Tr, Y

def build_phys_dataset(n_pairs, pts_per_pair, rng):
    B = np.empty((n_pairs * pts_per_pair, 2 * N_SENSORS), dtype=np.float32)
    Tr = np.empty((n_pairs * pts_per_pair, 4), dtype=np.float32)
    for k in range(n_pairs):
        i_in = rng.choice(train_idx)
        t_out = rng.uniform(t_star[i_in], t_max)
        s, e = k * pts_per_pair, (k + 1) * pts_per_pair
        B[s:e] = make_branch(i_in).astype(np.float32)
        Tr[s:e, 0] = rng.random(pts_per_pair)
        Tr[s:e, 1] = rng.random(pts_per_pair)
        Tr[s:e, 2] = norm(t_out, t_min, t_max)
        Tr[s:e, 3] = (t_out - t_star[i_in]) / dt_max
    return B, Tr

class DeepONet(torch.nn.Module):
    def __init__(self, sensors_in, width, depth):
        super().__init__()
        self.branches = torch.nn.ModuleList([mlp([sensors_in] + [width] * depth, torch.nn.ReLU) for _ in range(3)])
        self.trunks = torch.nn.ModuleList([mlp([4] + [width] * depth, torch.nn.ReLU) for _ in range(3)])
        self.bias = torch.nn.Parameter(torch.zeros(3))

    def forward(self, b_in, t_in):
        outs = []
        for i in range(3):
            b = self.branches[i](b_in)
            t = self.trunks[i](t_in)
            outs.append((b * t).sum(dim=1, keepdim=True) + self.bias[i])
        return torch.cat(outs, dim=1)

rng = np.random.default_rng(SEED)
print("building deeponet train set")
B_train, T_train, Y_train = build_dataset(N_PAIRS, POINTS_PER_PAIR, train_idx, rng)

don = DeepONet(2 * N_SENSORS, 128, 3).to(device)
don_opt = torch.optim.Adam(don.parameters(), lr=DON_LR)

Bt = torch.tensor(B_train, device=device)
Tt = torch.tensor(T_train, device=device)
Yt = torch.tensor(Y_train, device=device)

print("model 2/3, training deeponet on t 0 to 18")
for step in range(DON_ITERS + 1):
    idx = torch.randint(0, Bt.shape[0], (BATCH,))
    pred = don(Bt[idx], Tt[idx])
    loss = ((pred - Yt[idx]) ** 2).mean()
    don_opt.zero_grad()
    loss.backward()
    don_opt.step()
    if step % 2000 == 0:
        print(f"don step {step:5d} | data {loss.item():.3e}")

class FourierFeatureProjection(torch.nn.Module):
    def __init__(self, in_features, mapping_size=24, scale=4.0):
        super().__init__()
        self.register_buffer("B", torch.randn(in_features, mapping_size) * scale)

    def forward(self, x):
        proj = torch.matmul(x, self.B)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

class PIDeepONet(torch.nn.Module):
    def __init__(self, sensors_in, width, depth):
        super().__init__()
        self.rff = FourierFeatureProjection(4, mapping_size=24, scale=4.0)
        self.branches = torch.nn.ModuleList([mlp([sensors_in] + [width] * depth, torch.nn.Mish) for _ in range(3)])
        self.trunks = torch.nn.ModuleList([mlp([48] + [width] * depth, torch.nn.Mish) for _ in range(3)])
        self.bias = torch.nn.Parameter(torch.zeros(3))

    def forward(self, b_in, t_in):
        emb = self.rff(t_in)
        outs = []
        for i in range(3):
            b = self.branches[i](b_in)
            t = self.trunks[i](emb)
            outs.append((b * t).sum(dim=1, keepdim=True) + self.bias[i])
        return torch.cat(outs, dim=1)

def pi_residual(trunk_in_b, trunk_in_t):
    trunk_in_t = trunk_in_t.detach().clone().requires_grad_(True)
    out = pinet(trunk_in_b, trunk_in_t)
    u = out[:, 0:1] * U_SPAN + u_min
    v = out[:, 1:2] * V_SPAN + v_min
    p = out[:, 2:3] * P_SPAN + p_min

    def grad(y, x):
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y),
                                   create_graph=True, retain_graph=True)[0]

    du = grad(u, trunk_in_t); dv = grad(v, trunk_in_t); dp = grad(p, trunk_in_t)
    u_x = du[:, 0:1] / X_SPAN; u_y = du[:, 1:2] / Y_SPAN
    v_x = dv[:, 0:1] / X_SPAN; v_y = dv[:, 1:2] / Y_SPAN
    p_x = dp[:, 0:1] / X_SPAN; p_y = dp[:, 1:2] / Y_SPAN

    u_t = du[:, 2:3] / T_SPAN + du[:, 3:4] / dt_max
    v_t = dv[:, 2:3] / T_SPAN + dv[:, 3:4] / dt_max

    u_xx = grad(u_x, trunk_in_t)[:, 0:1] / X_SPAN
    u_yy = grad(u_y, trunk_in_t)[:, 1:2] / Y_SPAN
    v_xx = grad(v_x, trunk_in_t)[:, 0:1] / X_SPAN
    v_yy = grad(v_y, trunk_in_t)[:, 1:2] / Y_SPAN

    r_cont = u_x + v_y
    r_xmom = u_t + u * u_x + v * u_y + p_x - (1.0 / RE) * (u_xx + u_yy)
    r_ymom = v_t + u * v_x + v * v_y + p_y - (1.0 / RE) * (v_xx + v_yy)
    return r_cont, r_xmom, r_ymom

print("building physics set, t 0 to 20")
B_phys, T_phys = build_phys_dataset(N_PHYS_PAIRS, PHYS_PTS_PER_PAIR, rng)

pinet = PIDeepONet(2 * N_SENSORS, 128, 3).to(device)
Bp = torch.tensor(B_phys, device=device)
Tp = torch.tensor(T_phys, device=device)

sa_weights = torch.nn.Parameter(torch.ones(Bp.shape[0], 1, device=device))
pi_opt = torch.optim.Adam(pinet.parameters(), lr=PI_LR)
opt_sa = torch.optim.Adam([sa_weights], lr=SA_LR, maximize=True)

print("model 3/3, training sa-pi-deeponet, data on t 0 to 18, physics on t 0 to 20")
for step in range(PI_ITERS + 1):
    idx_d = torch.randint(0, Bt.shape[0], (BATCH,))
    idx_p = torch.randint(0, Bp.shape[0], (BATCH,))

    pred = pinet(Bt[idx_d], Tt[idx_d])
    loss_data = ((pred - Yt[idx_d]) ** 2).mean()

    rc, rx, ry = pi_residual(Bp[idx_p], Tp[idx_p])
    res_sq = rc ** 2 + rx ** 2 + ry ** 2
    w = sa_weights[idx_p] ** 2
    loss_phys = (w * res_sq).mean()

    loss = loss_data + loss_phys

    pi_opt.zero_grad()
    opt_sa.zero_grad()
    loss.backward()
    pi_opt.step()
    opt_sa.step()

    if step % 2000 == 0:
        print(f"pi step {step:5d} | data {loss_data.item():.3e} | phys(w) {loss_phys.item():.3e} | mean w {sa_weights.mean().item():.2f}")

def predict_pinn(i_out):
    coords = np.stack([x_n, y_n, np.full(N, t_n_all[i_out])], axis=1).astype(np.float32)
    with torch.no_grad():
        out = pinn_net(torch.tensor(coords, device=device)).cpu().numpy()
    return (denorm(out[:, 0], u_min, u_max),
            denorm(out[:, 1], v_min, v_max),
            denorm(out[:, 2], p_min, p_max))

def predict_operator(model, i_out, i_in):
    branch = np.tile(make_branch(i_in), (N, 1)).astype(np.float32)
    trunk = np.stack([x_n, y_n,
                      np.full(N, t_n_all[i_out]),
                      np.full(N, (t_star[i_out] - t_star[i_in]) / dt_max)], axis=1).astype(np.float32)
    with torch.no_grad():
        out = model(torch.tensor(branch, device=device), torch.tensor(trunk, device=device)).cpu().numpy()
    return (denorm(out[:, 0], u_min, u_max),
            denorm(out[:, 1], v_min, v_max),
            denorm(out[:, 2], p_min, p_max))

errs = {m: {f: [] for f in "uvp"} for m in ("pinn", "don", "pi")}
for i_out in range(T):
    i_in = 0 if i_out <= split_i else split_i
    preds = {
        "pinn": predict_pinn(i_out),
        "don": predict_operator(don, i_out, i_in),
        "pi": predict_operator(pinet, i_out, i_in),
    }
    for name, (pu, pv, pp) in preds.items():
        errs[name]["u"].append(rel_l2(pu, u_all[:, i_out]))
        errs[name]["v"].append(rel_l2(pv, v_all[:, i_out]))
        errs[name]["p"].append(rel_l2(pp, p_all[:, i_out]))

np.save("errs_train_to_18.npy", errs)

labels = {"pinn": "PINN", "don": "DeepONet", "pi": "SA-PI-DeepONet"}

print("\n" + "=" * 70)
print(f"  METRICS: {RUN_TAG} | mean L2 relative error (unitless)")
print("=" * 70)
print(f"{'model':<16}{'region':<28}{'u':>9}{'v':>9}{'p':>9}")
for name in ("pinn", "don", "pi"):
    for region, idxs in (("train (t = 0 to 18)", train_idx), ("generalization (t > 18)", test_idx)):
        row = [np.mean([errs[name][f][i] for i in idxs]) for f in "uvp"]
        print(f"{labels[name]:<16}{region:<28}{row[0]:>9.4f}{row[1]:>9.4f}{row[2]:>9.4f}")
print("=" * 70 + "\n")

model_colors = {"pinn": "tab:purple", "don": "tab:red", "pi": "tab:blue"}
field_names = {"u": "u (streamwise velocity)", "v": "v (transverse velocity)", "p": "p (pressure)"}

fig, axs = plt.subplots(1, 3, figsize=(16, 4.8), sharey=False)
for ax, f in zip(axs, "uvp"):
    for name in ("pinn", "don", "pi"):
        ax.plot(t_star, errs[name][f], color=model_colors[name], linewidth=2, label=labels[name])
    ax.axvline(TRAIN_T_MAX, color="red", ls=":", linewidth=1.5, label="end of training data (t = 18)")
    ax.set_title(field_names[f], fontsize=11)
    ax.set_xlabel("simulation time (nondimensional units)")
    ax.grid(True, alpha=0.3)
axs[0].set_ylabel("L2 relative error (prediction vs true, unitless)")
axs[0].legend(fontsize=9)
fig.suptitle("error accumulation over time, all models trained on t = 0 to 18", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RUN_TAG}_error_accumulation.png", dpi=200, bbox_inches="tight")
plt.show()

rows = []
for name in ("pinn", "don", "pi"):
    for region, idxs in (("train (t = 0 to 18)", train_idx), ("generalization (t > 18)", test_idx)):
        rows.append([labels[name], region] + [f"{np.mean([errs[name][f][i] for i in idxs]):.4f}" for f in "uvp"])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.axis("off")
tbl = ax.table(cellText=rows,
               colLabels=["model", "region", "u", "v", "p"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)
ax.set_title("mean L2 relative error (unitless), all models", fontsize=11, pad=10)
plt.savefig(f"{RUN_TAG}_metrics_table.png", dpi=200, bbox_inches="tight")
plt.show()